# 07 — Train Word2Vec Models (Stage 8, production pipeline)

**Does:** train one skip-gram model per (period x seed) from validated shards, with a manifest-tracked, resumable, checkable model store. This is the pipeline you were missing — it exists now.

**Reads:** `config/period_definitions.csv` (frozen v1.0), `shards/tokenized/*.jsonl.gz`, `config/project_config.yaml` v0.2.1. **Writes:** `models/word2vec/<corpus>/<group>/w2v__...model` + sidecar `.nfo.json` (full data provenance), per-epoch checkpoints, `manifests/training_manifest.csv`, `metadata/corpus_master.csv` (the one big metadata CSV), stability JSONs.

**Memory design:** shards stream as an iterator (never a list); vocabulary built in a separate pass; epochs reread shards from disk. One model in RAM at a time. Rerun-safe: COMPLETE rows skip by checksum; failed rows retry only.

In [ ]:
# Cell 1 — RUN PLAN (the only cell you must edit).
MODEL_FILTER_SUB = None   # e.g. 'AskAcademia' for a first proving run; None = all sufficient periods
PERIOD_FILTER = None        # e.g. ['2019Q1']; None = all
SEED_FILTER = None          # e.g. [1047]; None = all config seed_candidates
MAX_MODELS = 2              # pilot cap on (period x seed) trainings; None = uncapped production
DRY_RUN = False             # True = synthetic shards + throwaway 1-epoch dim-10 model in colab_tmp only
print(f"filter sub={MODEL_FILTER_SUB} periods={PERIOD_FILTER} seeds={SEED_FILTER} cap={MAX_MODELS} dry={DRY_RUN}")

In [ ]:
# Cell 2 — Setup: root, config, logger, gensim (installed only here, kept out of earlier notebooks).
import os, sys, csv, json, gzip, time, gc, hashlib, subprocess, datetime
from pathlib import Path
import yaml
ROOT = Path("/content/drive/MyDrive/reddit_embeddings_project")
if not ROOT.exists(): ROOT = Path("/home/user/reddit_embeddings_project")
cfg = yaml.safe_load(open(ROOT / "config/project_config.yaml", encoding="utf-8"))
h = hashlib.sha256()
with open(ROOT / "config/project_config.yaml", "rb") as f: h.update(f.read())
CFG_SHA = h.hexdigest()
E = cfg["embeddings"]
print("config", cfg["config_version"], f"dim={E['dim']} window={E['window']} neg={E['negative']} epochs={E['epochs']} min_count={E['min_count']}")
try:
    import gensim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "gensim"])
    import gensim
from gensim.models import Word2Vec
print("gensim", gensim.__version__)
import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/07_train__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("train"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)
for d in ["models/word2vec", "models/checkpoints", "metadata/corpus_statistics"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
T_COLS = ["model_id","corpus_type","subreddit_or_group","period_id","spec_hash","dim","window","sg",
 "negative","epochs","min_count","max_vocab","subsample","workers","lr","seed","vocab_size",
 "words_processed","epochs_done","train_secs","peak_ram_mb","model_path","vectors_path",
 "model_sha256","vectors_sha256","config_version","status","diagnostics_path"]
TMAN = ROOT / "manifests/training_manifest.csv"
if not TMAN.exists(): open(TMAN, "w", encoding="utf-8").write(",".join(T_COLS) + "\n")
def trows(): return list(csv.DictReader(open(TMAN, encoding="utf-8")))
def tupsert(row):
    rows = [r for r in trows() if not (r["model_id"] == row["model_id"] and r["seed"] == row["seed"])] + [row]
    tmp = TMAN.with_suffix(".tmp")
    f = open(tmp, "w", newline="", encoding="utf-8"); w = csv.DictWriter(f, fieldnames=T_COLS)
    w.writeheader(); w.writerows(rows); f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, TMAN)
def atomic_bytes(path: Path, data: bytes):
    tmp = path.with_suffix(".tmp"); path.parent.mkdir(parents=True, exist_ok=True)
    with open(tmp, "wb") as f: f.write(data); f.flush(); os.fsync(f.fileno())
    assert tmp.stat().st_size > 0; os.replace(tmp, path)
def sha_of(p: Path):
    hh = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): hh.update(b)
    return hh.hexdigest()
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
print("setup ready; training rows:", len(trows()))

In [ ]:
# Cell 3 — THE BIG METADATA CSV: compile metadata/corpus_master.csv (monthly grain).
# Joins monthly_counts + period assignment (date-range lookup) + shard coverage (min/max ts overlap).
# This single file is the checkable record of what data every model stands on. Rebuilt idempotently.
import glob
PDEF = ROOT / "config/period_definitions.csv"
periods = [r for r in csv.DictReader(open(PDEF, encoding="utf-8")) if not r["model_id"].startswith("#")] if PDEF.exists() else []
print(f"period definitions: {len(periods)}")
def period_for(sub, mm):
    probe = mm + "-15"
    for r in periods:
        if r["subreddit_or_group"] == sub and r["start_date"][:7] <= mm and probe <= r["end_date"]:
            return r["model_id"], r["sufficiency"]
    return "", ""
shard_months = {}
SMAN = ROOT / "manifests/shard_manifest.csv"
if SMAN.exists():
    for r in csv.DictReader(open(SMAN, encoding="utf-8")):
        try:
            m0, m1 = r["min_ts"][:7], r["max_ts"][:7]
            shard_months[(r["subreddit"], m0)] = shard_months.get((r["subreddit"], m0), 0) + 1
            if m1 != m0: shard_months[(r["subreddit"], m1)] = shard_months.get((r["subreddit"], m1), 0) + 1
        except Exception: pass
files = sorted(glob.glob(str(ROOT / "metadata/monthly_counts/*.csv")))
out_lines, n = [], 0
for f in files:
    try: rows = list(csv.DictReader(open(f, encoding="utf-8")))
    except Exception as e: lg.error(f"unreadable {f}: {e}"); continue
    for r in rows:
        pid, suf = period_for(r["subreddit"], r["month"])
        r["period_id"], r["sufficiency"] = pid, suf
        r["shards_present"] = shard_months.get((r["subreddit"], r["month"]), 0)
        r["master_built"] = today; r["config_version"] = cfg["config_version"]
        out_lines.append(r); n += 1
MASTER = ROOT / "metadata/corpus_master.csv"
if out_lines:
    cols = list(out_lines[0].keys())
    tmp = MASTER.with_suffix(".tmp")
    f = open(tmp, "w", newline="", encoding="utf-8"); w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader(); w.writerows(out_lines); f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, MASTER)
    print(f"corpus_master.csv: {n} rows from {len(files)} monthly files -> {MASTER}")
else:
    print("no monthly counts yet — master CSV deferred (run Notebook 02 first; DRY_RUN still works)")

In [ ]:
# Cell 4 — REGISTRY: expected models = sufficient periods x seeds. Status from training_manifest + file check.
# A model file counts as COMPLETE only if: manifest says complete + file exists + sha matches.
wanted = []
for r in periods:
    if r["sufficiency"] not in ("sufficient", "axis_grade", "marginal_merge_first"): continue
    if MODEL_FILTER_SUB and r["subreddit_or_group"] != MODEL_FILTER_SUB: continue
    if PERIOD_FILTER and r["model_id"] not in PERIOD_FILTER and r["start_date"][:7] not in PERIOD_FILTER: continue
    for s in (SEED_FILTER or cfg["embeddings"]["seed_candidates"]):
        wanted.append((r, int(s)))
if MAX_MODELS: wanted = wanted[:MAX_MODELS]
print(f"wanted trainings: {len(wanted)}")
def store_dir(corpus, group): return ROOT / "models/word2vec" / corpus / str(group).lower()
def stem_of(group, period, seed):
    g = str(group).lower(); g = "".join(c for c in g if c.isalnum())
    per = period.replace("__", "_").split("__")[-1] if "__" in period else period
    return f"w2v__{g}__{per}__cfg-{cfg['config_version']}__seed-{seed}"
reg = []
known = {(r["model_id"], r["seed"]): r for r in trows()}
for r, s in wanted:
    mid = r["model_id"]
    mp = store_dir(r["corpus_type"], r["subreddit_or_group"]) / (stem_of(r["subreddit_or_group"], mid, s) + ".model")
    st = (known.get((mid, str(s)), {}) or {}).get("status", "pending")
    ok = st == "complete" and mp.exists() and mp.stat().st_size > 0
    if st == "complete" and not ok: st = "validation_failed(file_missing_or_empty)"
    reg.append((mid, s, st, str(mp) if ok else "-"))
from collections import Counter
print(Counter(s for _, _, s, _ in reg))
for mid, s, st, _ in reg[:20]: print(f"  {st:12s} {mid} seed={s}")
todo = [(m, s) for m, s, st, _ in reg for m, s in [r for r, ss in wanted if r["model_id"] == m and ss == s] if st != "complete"]
print(f"to train: {len(todo)}")

In [ ]:
# Cell 5 — STREAMING CORPUS + TRAIN LOOP. One model in RAM at a time; checkpoints per epoch.
class ShardSentences:
    """Re-iterable stream of token lists over validated .jsonl.gz shards. Never a list."""
    def __init__(self, paths): self.paths = list(paths)
    def __iter__(self):
        for p in self.paths:
            with gzip.open(p, "rt", encoding="utf-8") as f:
                for line in f:
                    if not line or line[0] == "#": continue
                    try: rec = json.loads(line)
                    except Exception: continue
                    toks = rec.get("tokens")
                    if isinstance(toks, list) and len(toks) >= 3: yield toks
def shards_for(sub, pid):
    base = ROOT / "shards/tokenized"
    return sorted(base.rglob(f"*{str(sub).lower()}*{pid}*.jsonl.gz")) + \
           sorted(base.rglob(f"*{str(sub).lower()}*.jsonl.gz"))
def spec_hash(d): return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:12]
def train_one(r, seed):
    mid = r["model_id"]
    grp, corpus = r["subreddit_or_group"], r["corpus_type"]
    sdir = store_dir(corpus, grp); stem = stem_of(grp, mid, seed)
    mpath = sdir / (stem + ".model")
    spec = {"dim": E["dim"], "window": E["window"], "sg": 1, "negative": E["negative"],
            "epochs": E["epochs"], "min_count": E["min_count"], "sample": E["subsample"]}
    row = {"model_id": mid, "corpus_type": corpus, "subreddit_or_group": grp, "period_id": mid,
           "spec_hash": spec_hash(spec), "dim": E["dim"], "window": E["window"], "sg": 1,
           "negative": E["negative"], "epochs": E["epochs"], "min_count": E["min_count"],
           "max_vocab": E["max_vocab"], "subsample": E["subsample"], "workers": E["workers"],
           "lr": str(E["lr"]), "seed": seed, "vocab_size": "", "words_processed": "",
           "epochs_done": 0, "train_secs": "", "peak_ram_mb": "", "model_path": str(mpath),
           "vectors_path": "", "model_sha256": "", "vectors_sha256": "",
           "config_version": cfg["config_version"], "status": "in_progress", "diagnostics_path": ""}
    tupsert(row)
    try:
        paths = shards_for(grp, mid)
        assert paths, f"no shards found for {grp}/{mid} — run Notebook 04/05 first"
        sents = ShardSentences(paths)
        t0 = time.time()
        model = Word2Vec(vector_size=E["dim"], window=E["window"], sg=1, negative=E["negative"],
                         min_count=E["min_count"], sample=E["subsample"], workers=E["workers"],
                         seed=int(seed), epochs=1)
        model.build_vocab(sents)  # pass 1: vocabulary only
        for ep in range(E["epochs"]):  # passes 2..N: reread shards each epoch
            model.train(ShardSentences(paths), total_examples=model.corpus_count, epochs=1)
            ck = ROOT / "models/checkpoints" / (stem + f".ep{ep + 1}.model")
            ctmp = ck.with_suffix(".tmp"); model.save(str(ctmp)); os.replace(ctmp, ck)
            row["epochs_done"] = ep + 1; tupsert(row)
        tmp = mpath.with_suffix(".tmp"); model.save(str(tmp)); os.replace(tmp, mpath)
        assert mpath.stat().st_size > 0 and Word2Vec.load(str(mpath)) is not None
        try:
            import resource; peak = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
        except Exception: peak = "n/a"
        nfo = {"model_id": mid, "seed": seed, "spec": spec, "config_version": cfg["config_version"],
               "config_sha256": CFG_SHA, "gensim": gensim.__version__, "sys": sys.version.split()[0],
               "shards": [str(p) for p in paths], "shard_sha256": {str(p): sha_of(p) for p in paths},
               "vocab_size": len(model.wv), "corpus_count": model.corpus_count,
               "train_secs": round(time.time() - t0, 1), "period_row": r}
        atomic_bytes(sdir / (stem + ".nfo.json"), json.dumps(nfo, indent=2).encode())
        row.update({"status": "complete", "vocab_size": len(model.wv), "words_processed": model.corpus_total_words,
            "train_secs": nfo["train_secs"], "peak_ram_mb": peak, "model_sha256": sha_of(mpath),
            "diagnostics_path": ""}); tupsert(row)
        print(f"OK {stem}: vocab={len(model.wv)} words={model.corpus_total_words} secs={nfo['train_secs']}")
        del model; gc.collect()
        return True
    except Exception as e:
        lg.exception(mid)
        row.update({"status": "failed"}); tupsert(row)
        with open(ROOT / "manifests/failure_log.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps({"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(), "stage": "train",
                "unit": mid, "seed": seed, "error": str(e)[:300], "config_version": cfg["config_version"]}) + "\n")
        print(f"FAIL {mid} seed={seed}: {str(e)[:140]}"); return False
print("train fn ready")

In [ ]:
# Cell 6 — EXECUTE (or DRY_RUN synthetic proof). Production trains todo[] in order; crash loses <=1 model.
if DRY_RUN:
    import random
    random.seed(7); vocab = [f"w{i}" for i in range(60)] + ["not", "never", "good", "bad"]
    dp = Path(cfg["paths"]["colab_tmp"] if "colab_tmp" in cfg["paths"] else "/tmp") / "dry_shards"
    dp.mkdir(parents=True, exist_ok=True)
    sp = dp / "dry__0000.jsonl.gz"
    f.write('#manifest {"dry": true}' + chr(10))
    for i in range(600):
        f.write(json.dumps({"tokens": [random.choice(vocab) for _ in range(random.randint(5, 25))]}) + "\\n")
    f.close()
    m = Word2Vec(vector_size=10, window=3, sg=1, negative=5, min_count=2, workers=1, seed=7, epochs=1)
    m.build_vocab(ShardSentences([sp])); m.train(ShardSentences([sp]), total_examples=m.corpus_count, epochs=1)
    assert len(m.wv) > 10 and "not" in m.wv, "dry-run model invalid"
    print(f"DRY_RUN PASS: vocab={len(m.wv)} (negation kept: {'not' in m.wv}); nothing written to the model store")
    ok = fail = 0
else:
    if not periods: raise SystemExit("STOP: period_definitions.csv is empty — run Notebook 03 first (no silent training without frozen periods).")
    ok = fail = 0
    for r, s in todo:
        if train_one(r, s): ok += 1
        else: fail += 1
    print(f"trained ok={ok} failed={fail}")

In [ ]:
# Cell 7 — VALIDATE each new model: reopen, vocab/neighbors of top-frequency probes (data-driven, no concepts).
import numpy as np
for (mid, s, st, mp) in reg:
    if st == "complete" or DRY_RUN: continue
    known = {(r["model_id"], r["seed"]): r for r in trows()}.get((mid, str(s)))
    if not known or known["status"] != "complete": continue
    try:
        m = Word2Vec.load(mp)
        counts = [(w, m.wv.get_vecattr(w, "count")) for w in m.wv.index_to_key]
        counts.sort(key=lambda x: -x[1])
        probes = [w for w, _ in counts[:8]]
        dg = {"model_id": mid, "seed": s, "vocab": len(m.wv), "top": probes,
              "negation_kept": "not" in m.wv,
              "neighbors": {w: [x for x, _ in m.wv.most_similar(w, topn=5)] for w in probes[:3]}}
        dp = ROOT / "diagnostics/model_stability" / (Path(mp).stem + ".json")
        atomic_bytes(dp, json.dumps(dg, indent=2).encode())
        r2 = dict(known); r2["diagnostics_path"] = str(dp); tupsert(r2)
        print(f"VALID {Path(mp).stem}: vocab={len(m.wv)} negation_kept={dg['negation_kept']}")
        del m; gc.collect()
    except Exception as e:
        lg.error(f"validate {mid}: {e}"); print(f"VALIDATE-FAIL {mid}: {str(e)[:120]}")

In [ ]:
# Cell 8 — REGISTRY REPRINT + END-OF-RUN SUMMARY.
from collections import Counter
final = Counter(({(r['model_id'], r['seed']): r for r in trows()}.get((m, str(s)), {}) or {}).get("status", "pending") for m, s in [(mm, ss) for rr, ss in wanted for mm in [rr["model_id"]]])
print(dict(final))
print("=" * 70)
print(f"TRAINING {'COMPLETE' if fail == 0 else 'COMPLETE-WITH-FAILURES'} ok={ok} failed={fail}")
print("store     : models/word2vec/<corpus>/<group>/w2v__<grp>__<period>__cfg-<ver>__seed-<s>.model + .nfo.json")
print("metadata  : metadata/corpus_master.csv (big CSV) + manifests/training_manifest.csv (registry)")
print("remaining : failed rows only on rerun; then 08 seeds -> 09 normalize -> 10 inspect")
print("rerun safe: YES (complete+checksum skips; checkpoints resume at model granularity)")
print("next      : 08_evaluate_random_seeds.ipynb (or 09/10 once built)")
print("=" * 70)